# Reusable GPU runtime — URL and private token
Select a GPU runtime and run the cell. It starts a token-protected localhost execution service behind a temporary HTTPS tunnel. The token grants runtime code, shell, terminal and file access. Google Drive is not mounted.
Blender, FFmpeg, NVIDIA graphics selection and the licensed Snow asset are cached once per runtime. No particular video is embedded or automatically rendered. Upload new projects, build scenes and run small GPU tests through the same connection.
A live dashboard refreshes every 3 seconds with combined completed frames, worker activity, system RAM, stage, measured throughput and ETA. New EEVEE jobs request up to four persistent workers, selectable per job from 1 to 4. Host RAM admission limits scene copies (normally two on a 12.7 GiB VM). Four workers exhausted host RAM in the first live run.
Rerunning in the same live VM reuses the authenticated server/jobs and refreshes its tunnel. A new/deleted runtime has a new token and loses ephemeral files. Colab limits and external-control restrictions still apply; uptime and free GPU assignment are not guaranteed.
Snow Rig © Blender Foundation | studio.blender.org — CC BY 4.0.

In [ ]:
# Reusable runtime bridge: run once, then send jobs through the URL/token.
import base64,hashlib,io,json,os,pathlib,platform,re,secrets,socket,subprocess,sys,threading,time,urllib.request,zipfile
subprocess.run([sys.executable,'-m','pip','install','-q','fastapi>=0.110,<1','uvicorn>=0.24,<1','psutil>=5.9,<8','requests>=2.31,<3'],check=True)
import requests,psutil
PAYLOAD = '...'
EXPECTED_SHA256 = '8fb0d5461a915945f5c28ea189e1d28473971aeec6b1154ac9e2bb3befcb6489'
server_source = '"""Colab Codex Bridge: authenticated localhost execution API.\n\nNever expose\nthis server on 0.0.0.0; tunnel localhost only. The bearer token grants remote\nshell execution with the permissions of the Colab runtime.\n"""\nfrom __future__ import annotations\n\nimport base64\nimport hashlib\nimport collections\nimport json\nimport os\nimport pathlib\nimport secrets\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\nimport traceback\nimport uuid\nfrom datetime import datetime, timezone\nfrom typing import Optional\n\nimport psutil\nfrom fastapi import Depends, FastAPI, Header, HTTPException, Query\nfrom pydantic import BaseModel, Field\n\nBRIDGE_DIR = pathlib.Path(os.environ.get(\'COLAB_BRIDGE_DIR\', \'/content/colab_codex_bridge\')).resolve()\nSTATE_DIR = BRIDGE_DIR / \'state\'\nJOB_DIR = BRIDGE_DIR / \'jobs\'\nPERSIST_DIR = pathlib.Path(os.environ.get(\'COLAB_BRIDGE_PERSIST_DIR\', \'\')).expanduser() if os.environ.get(\'COLAB_BRIDGE_PERSIST_DIR\') else None\nTOKEN = os.environ.get(\'COLAB_BRIDGE_TOKEN\', \'\')\nSTARTED_AT = time.time()\nMAX_FILE = 4 * 1024 * 1024\nMAX_LOG_BYTES = 160000\nLOCK = threading.RLock()\nPROCESSES = {}\nJOBS = {}\nEVENTS = collections.deque(maxlen=100)\nTERMINALS = {}\nENGINE_ROOT = pathlib.Path(os.environ.get(\'VIDEO_ENGINE_ROOT\',\'/content/video_engine\')).resolve()\n\ndef job_environment():\n    env={k:v for k,v in os.environ.items() if k!=\'COLAB_BRIDGE_TOKEN\'}\n    config=ENGINE_ROOT/\'gpu-environment.json\'\n    if config.is_file():\n        values=json.loads(config.read_text())\n        env.update({k:v for k,v in values.items() if k in (\'__EGL_VENDOR_LIBRARY_FILENAMES\',\'__GLX_VENDOR_LIBRARY_NAME\',\'LD_LIBRARY_PATH\')})\n    return env\n\nfor d in (BRIDGE_DIR, STATE_DIR, JOB_DIR):\n    d.mkdir(parents=True, exist_ok=True)\n\nif not TOKEN or len(TOKEN) < 32:\n    raise RuntimeError(\'COLAB_BRIDGE_TOKEN must be set to a long random secret\')\n\napp = FastAPI(title=\'Colab Codex Bridge\', docs_url=None, redoc_url=None, openapi_url=None)\n\n\ndef now():\n    return datetime.now(timezone.utc).isoformat(timespec=\'seconds\')\n\n\ndef event(event_type: str, **values):\n    with LOCK:\n        EVENTS.append({\'at\': now(), \'event\': event_type, **values})\n\n\ndef put_json(path: pathlib.Path, data):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.\' + secrets.token_hex(4) + \'.tmp\')\n    with temporary.open(\'w\', encoding=\'utf-8\') as f:\n        json.dump(data, f, ensure_ascii=False, indent=2)\n        f.flush()\n    temporary.replace(path)\n\n\ndef secure(authorization: Optional[str] = Header(default=None)):\n    supplied = (authorization or \'\').removeprefix(\'Bearer \').strip()\n    if not secrets.compare_digest(supplied, TOKEN):\n        raise HTTPException(status_code=401, detail=\'Invalid bearer token\')\n\n\ndef gpu_details():\n    command = [\n        \'nvidia-smi\',\n        \'--query-gpu=index,name,driver_version,utilization.gpu,memory.used,memory.total,temperature.gpu\',\n        \'--format=csv,noheader,nounits\'\n    ]\n    try:\n        result = subprocess.run(command, capture_output=True, text=True, timeout=5)\n        if result.returncode != 0:\n            return {\'available\': False, \'error\': result.stderr.strip()[:500]}\n        rows = []\n        for line in result.stdout.splitlines():\n            cols = [x.strip() for x in line.split(\',\')]\n            if len(cols) >= 7:\n                rows.append(dict(zip([\'index\', \'name\', \'driver\', \'utilization_percent\', \'memory_used_mib\',\n                                      \'memory_total_mib\', \'temperature_c\'], cols[:7])))\n        return {\'available\': bool(rows), \'devices\': rows}\n    except (OSError, subprocess.TimeoutExpired) as exc:\n        return {\'available\': False, \'error\': f\'{type(exc).__name__}: {exc}\'}\n\n\ndef job_with_progress(job):\n    result=job.copy()\n    path=JOB_DIR/job[\'job_id\']/\'output.log\'\n    try:\n        with path.open(\'rb\') as stream:\n            stream.seek(0,2);size=stream.tell();stream.seek(max(0,size-262144))\n            lines=stream.read().split(b\'\\n\')[:-1]\n        for line in reversed(lines):\n            if line.startswith((b\'RENDER_PROGRESS \',b\'ENGINE_PROGRESS \')):\n                try:progress=json.loads(line.split(b\' \',1)[1])\n                except (ValueError,UnicodeError):continue\n                if isinstance(progress,dict):result[\'progress\']=progress;break\n    except OSError:pass\n    return result\n\n\ndef status():\n    ram = psutil.virtual_memory()\n    disk = psutil.disk_usage(str(BRIDGE_DIR))\n    with LOCK:\n        running = [j[\'job_id\'] for j in JOBS.values() if j[\'status\'] == \'running\']\n        recent = list(JOBS.values())[-12:]\n        events = list(EVENTS)[-15:]\n    return {\n        \'online\': True,\n        \'timestamp_utc\': now(),\n        \'uptime_seconds\': round(time.time() - STARTED_AT),\n        \'hostname\': os.uname().nodename if hasattr(os, \'uname\') else \'unknown\',\n        \'python\': sys.version.split()[0],\n        \'gpu\': gpu_details(),\n        \'cpu_percent\': psutil.cpu_percent(interval=None),\n        \'memory\': {\'total_gb\': round(ram.total / 2**30, 2), \'used_gb\': round(ram.used / 2**30, 2), \'available_gb\': round(ram.available / 2**30, 2)},\n        \'disk\': {\'free_gb\': round(disk.free / 2**30, 2), \'total_gb\': round(disk.total / 2**30, 2)},\n        \'bridge_dir\': str(BRIDGE_DIR),\n        \'persistent_state_dir\': str(PERSIST_DIR) if PERSIST_DIR else None,\n        \'active_job_ids\': running,\n        \'recent_jobs\': [{k: j.get(k) for k in (\'job_id\', \'kind\', \'status\', \'exit_code\', \'started_at\', \'finished_at\', \'command\',\'progress\')}\n                        for j in map(job_with_progress,recent)],\n        \'events\': events,\n    }\n\n\ndef persist_snapshot():\n    snapshot = status()\n    put_json(STATE_DIR / \'last_status.json\', snapshot)\n    if PERSIST_DIR:\n        try:\n            PERSIST_DIR.mkdir(parents=True, exist_ok=True)\n            put_json(PERSIST_DIR / \'last_status.json\', snapshot)\n            with LOCK:\n                jobs = list(JOBS.values())[-30:]\n            put_json(PERSIST_DIR / \'jobs.json\', jobs)\n            for job in jobs:\n                source = JOB_DIR / job[\'job_id\'] / \'output.log\'\n                if source.exists():\n                    target = PERSIST_DIR / \'logs\' / f"{job[\'job_id\']}.log"\n                    target.parent.mkdir(parents=True, exist_ok=True)\n                    with source.open(\'rb\') as reader:\n                        reader.seek(max(0, source.stat().st_size - MAX_LOG_BYTES))\n                        data = reader.read()\n                    target.write_bytes(data)\n            (PERSIST_DIR / \'heartbeat.txt\').write_text(now(), encoding=\'utf-8\')\n        except Exception as exc:\n            event(\'persistence_error\', detail=f\'{type(exc).__name__}: {exc}\'[:300])\n    return snapshot\n\n\ndef background_snapshots():\n    while True:\n        try:\n            persist_snapshot()\n        except Exception as exc:\n            event(\'snapshot_error\', detail=f\'{type(exc).__name__}: {exc}\'[:300])\n        time.sleep(15)\n\n\ndef wait_for_exit(job_id: str, process: subprocess.Popen):\n    rc = process.wait()\n    with LOCK:\n        j = JOBS[job_id]\n        j[\'status\'] = \'cancelled\' if j.get(\'cancel_requested\') else (\'succeeded\' if rc == 0 else \'failed\')\n        j[\'exit_code\'] = rc\n        j[\'finished_at\'] = now()\n        put_json(JOB_DIR / job_id / \'state.json\', j)\n        PROCESSES.pop(job_id, None)\n        event(\'job_finished\', job_id=job_id, exit_code=rc, state=j[\'status\'])\n    try:\n        persist_snapshot()\n    except Exception:\n        pass\n\n\ndef launch(command: str, kind: str, cwd: str, timeout: int, python_code: Optional[str] = None):\n    job_id = str(uuid.uuid4())\n    workdir = pathlib.Path(cwd).expanduser().resolve()\n    if not workdir.is_dir():\n        raise HTTPException(status_code=400, detail=f\'Working directory not found: {workdir}\')\n    folder = JOB_DIR / job_id\n    folder.mkdir(parents=True, exist_ok=True)\n    if kind == \'python\':\n        script = folder / \'task.py\'\n        script.write_text(python_code or \'\', encoding=\'utf-8\')\n        argv = [sys.executable, \'-u\', str(script)]\n        command_summary = f\'python {script} (code length {len(python_code or "")})\'\n    else:\n        argv = [\'/bin/bash\', \'-lc\', command]\n        command_summary = command[:700]\n    job = {\n        \'job_id\': job_id,\n        \'kind\': kind,\n        \'command\': command_summary,\n        \'cwd\': str(workdir),\n        \'started_at\': now(),\n        \'finished_at\': None,\n        \'status\': \'running\',\n        \'exit_code\': None,\n        \'timeout_seconds\': timeout,\n    }\n    try:\n        output_handle = (folder / \'output.log\').open(\'wb\')\n        process = subprocess.Popen(argv, cwd=str(workdir), stdout=output_handle,\n                                   stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,\n                                   start_new_session=True, bufsize=0,\n                                   env=job_environment())\n        output_handle.close()\n    except Exception as exc:\n        raise HTTPException(status_code=500, detail=f\'Failed to start: {exc}\')\n    with LOCK:\n        JOBS[job_id] = job\n        PROCESSES[job_id] = process\n        put_json(folder / \'state.json\', job)\n        event(\'job_started\', job_id=job_id, kind=kind)\n    threading.Thread(target=wait_for_exit, args=(job_id, process), daemon=True).start()\n    if timeout > 0:\n        def limit():\n            try:\n                process.wait(timeout=timeout)\n            except subprocess.TimeoutExpired:\n                with LOCK:\n                    if job_id in JOBS:\n                        JOBS[job_id][\'timeout_reached\'] = True\n                try:\n                    os.killpg(process.pid, signal.SIGTERM)\n                except ProcessLookupError:\n                    pass\n        threading.Thread(target=limit, daemon=True).start()\n    return job\n\n\nclass CommandRequest(BaseModel):\n    command: str = Field(min_length=1, max_length=100000)\n    cwd: str = \'/content\'\n    timeout_seconds: int = Field(default=3600, ge=0, le=86400)\n\n\nclass PythonRequest(BaseModel):\n    code: str = Field(min_length=1, max_length=2000000)\n    cwd: str = \'/content\'\n    timeout_seconds: int = Field(default=3600, ge=0, le=86400)\n\n\nclass WriteRequest(BaseModel):\n    path: str\n    content_base64: str\n    create_parents: bool = True\n\nclass ChunkRequest(BaseModel):\n    path: str\n    content_base64: str\n    offset: int = Field(ge=0)\n\nclass CommitRequest(BaseModel):\n    temporary: str\n    path: str\n    sha256: str = Field(pattern=\'^[0-9a-f]{64}$\')\n    size: int = Field(ge=0)\n\nclass TerminalWrite(BaseModel):\n    text: str = Field(max_length=100000)\n\n@app.get(\'/engine/capabilities\',dependencies=[Depends(secure)])\ndef engine_capabilities():\n    path=ENGINE_ROOT/\'runtime-capabilities.json\'\n    return json.loads(path.read_text()) if path.is_file() else {\'ready\':False,\'root\':str(ENGINE_ROOT)}\n\n@app.get(\'/files/stat\',dependencies=[Depends(secure)])\ndef file_stat(path: str):\n    p=pathlib.Path(path).expanduser().resolve()\n    if not p.is_file():raise HTTPException(status_code=404,detail=\'File not found\')\n    with p.open(\'rb\') as stream:digest=hashlib.file_digest(stream,\'sha256\').hexdigest()\n    return {\'path\':str(p),\'size_bytes\':p.stat().st_size,\'sha256\':digest}\n\n@app.get(\'/files/read_chunk\',dependencies=[Depends(secure)])\ndef read_chunk(path: str,offset:int=Query(default=0,ge=0),length:int=Query(default=3000000,ge=1,le=3000000)):\n    p=pathlib.Path(path).expanduser().resolve()\n    if not p.is_file():raise HTTPException(status_code=404,detail=\'File not found\')\n    with p.open(\'rb\') as stream:stream.seek(offset);data=stream.read(length)\n    return {\'content_base64\':base64.b64encode(data).decode(),\'next_offset\':offset+len(data),\'size_bytes\':p.stat().st_size}\n\n@app.post(\'/files/write_chunk\',dependencies=[Depends(secure)])\ndef write_chunk(req:ChunkRequest):\n    try:data=base64.b64decode(req.content_base64,validate=True)\n    except Exception:raise HTTPException(status_code=400,detail=\'Invalid base64\')\n    if len(data)>3000000:raise HTTPException(status_code=413,detail=\'Chunk exceeds 3 MB\')\n    p=pathlib.Path(req.path).expanduser().resolve();p.parent.mkdir(parents=True,exist_ok=True)\n    with LOCK:\n        size=p.stat().st_size if p.exists() else 0\n        if size!=req.offset:raise HTTPException(status_code=409,detail=\'Upload offset differs from current file size\')\n        with p.open(\'ab\') as stream:stream.write(data)\n    return {\'next_offset\':req.offset+len(data)}\n\n@app.post(\'/files/commit\',dependencies=[Depends(secure)])\ndef commit_file(req:CommitRequest):\n    p=pathlib.Path(req.temporary).expanduser().resolve();target=pathlib.Path(req.path).expanduser().resolve()\n    if not p.is_file():raise HTTPException(status_code=404,detail=\'Temporary upload not found\')\n    with p.open(\'rb\') as stream:digest=hashlib.file_digest(stream,\'sha256\').hexdigest()\n    if p.stat().st_size!=req.size or digest!=req.sha256:raise HTTPException(status_code=400,detail=\'Upload hash/size mismatch\')\n    target.parent.mkdir(parents=True,exist_ok=True);p.replace(target)\n    return {\'path\':str(target),\'sha256\':digest,\'size_bytes\':req.size}\n\n@app.post(\'/terminals\',dependencies=[Depends(secure)])\ndef start_terminal():\n    if os.name!=\'posix\':raise HTTPException(status_code=501,detail=\'PTY terminals require Linux\')\n    import pty\n    master,slave=pty.openpty();env=job_environment();env.update(TERM=\'xterm-256color\',PS1=\'gpu-runtime> \')\n    process=subprocess.Popen([\'/bin/bash\',\'--noprofile\',\'--norc\',\'-i\'],stdin=slave,stdout=slave,stderr=slave,start_new_session=True,env=env,cwd=str(ENGINE_ROOT if ENGINE_ROOT.is_dir() else BRIDGE_DIR))\n    os.close(slave);identifier=str(uuid.uuid4());folder=BRIDGE_DIR/\'terminals\'/identifier;folder.mkdir(parents=True)\n    entry={\'process\':process,\'master\':master,\'log\':folder/\'output.log\'}\n    with LOCK:TERMINALS[identifier]=entry\n    def read_output():\n        with entry[\'log\'].open(\'ab\',buffering=0) as output:\n            try:\n                while data:=os.read(master,4096):output.write(data)\n            except OSError:pass\n        process.wait()\n        try:os.close(master)\n        except OSError:pass\n    threading.Thread(target=read_output,daemon=True).start()\n    return {\'terminal_id\':identifier,\'pid\':process.pid}\n\n@app.get(\'/terminals/{identifier}/read\',dependencies=[Depends(secure)])\ndef read_terminal(identifier:str,offset:int=Query(default=0,ge=0),limit:int=Query(default=30000,ge=1,le=MAX_LOG_BYTES)):\n    with LOCK:entry=TERMINALS.get(identifier)\n    if entry is None:raise HTTPException(status_code=404,detail=\'Unknown terminal\')\n    data=b\'\'\n    if entry[\'log\'].is_file():\n        with entry[\'log\'].open(\'rb\') as stream:stream.seek(offset);data=stream.read(limit)\n    return {\'text\':data.decode(\'utf-8\',errors=\'replace\'),\'next_offset\':offset+len(data),\'running\':entry[\'process\'].poll() is None}\n\n@app.post(\'/terminals/{identifier}/write\',dependencies=[Depends(secure)])\ndef write_terminal(identifier:str,req:TerminalWrite):\n    with LOCK:entry=TERMINALS.get(identifier)\n    if entry is None:raise HTTPException(status_code=404,detail=\'Unknown terminal\')\n    if entry[\'process\'].poll() is not None:raise HTTPException(status_code=409,detail=\'Terminal has exited\')\n    try:os.write(entry[\'master\'],req.text.encode())\n    except OSError:raise HTTPException(status_code=409,detail=\'Terminal is closed\')\n    return {\'ok\':True}\n\n@app.post(\'/terminals/{identifier}/close\',dependencies=[Depends(secure)])\ndef close_terminal(identifier:str):\n    with LOCK:entry=TERMINALS.get(identifier)\n    if entry is None:raise HTTPException(status_code=404,detail=\'Unknown terminal\')\n    if entry[\'process\'].poll() is None:\n        try:os.killpg(entry[\'process\'].pid,signal.SIGHUP)\n        except ProcessLookupError:pass\n    return {\'ok\':True}\n\n\n@app.get(\'/health\', dependencies=[Depends(secure)])\ndef health():\n    return {\'status\': \'ok\', \'timestamp_utc\': now(), \'api_version\': 3}\n\n@app.on_event(\'shutdown\')\ndef shutdown_children():\n    with LOCK:\n        children=list(PROCESSES.values())+[entry[\'process\'] for entry in TERMINALS.values()]\n        for job in JOBS.values():\n            if job[\'status\']==\'running\':job[\'cancel_requested\']=True\n    for process in children:\n        if process.poll() is None:\n            try:\n                if os.name==\'posix\':os.killpg(process.pid,signal.SIGTERM)\n                else:process.terminate()\n            except ProcessLookupError:pass\n\n\n@app.get(\'/status\', dependencies=[Depends(secure)])\ndef read_status():\n    s = status()\n    put_json(STATE_DIR / \'last_status.json\', s)\n    return s\n\n\n@app.post(\'/jobs/command\', dependencies=[Depends(secure)])\ndef command_job(req: CommandRequest):\n    return launch(req.command, \'shell\', req.cwd, req.timeout_seconds)\n\n\n@app.post(\'/jobs/python\', dependencies=[Depends(secure)])\ndef python_job(req: PythonRequest):\n    return launch(\'\', \'python\', req.cwd, req.timeout_seconds, python_code=req.code)\n\n\n@app.get(\'/jobs\', dependencies=[Depends(secure)])\ndef list_jobs(limit: int = Query(default=20, ge=1, le=100)):\n    with LOCK:\n        return [job_with_progress(j) for j in list(JOBS.values())[-limit:]]\n\n\n@app.get(\'/jobs/{job_id}\', dependencies=[Depends(secure)])\ndef get_job(job_id: str):\n    with LOCK:\n        if job_id not in JOBS:\n            raise HTTPException(status_code=404, detail=\'Unknown job ID\')\n        return job_with_progress(JOBS[job_id])\n\n\n@app.get(\'/jobs/{job_id}/output\', dependencies=[Depends(secure)])\ndef job_output(job_id: str, offset: int = Query(default=0, ge=0), limit: int = Query(default=30000, ge=1, le=MAX_LOG_BYTES)):\n    with LOCK:\n        if job_id not in JOBS:\n            raise HTTPException(status_code=404, detail=\'Unknown job ID\')\n        job = job_with_progress(JOBS[job_id])\n    p = JOB_DIR / job_id / \'output.log\'\n    size = p.stat().st_size if p.exists() else 0\n    with p.open(\'rb\') as f:\n        f.seek(min(offset, size))\n        chunk = f.read(limit)\n    return {\'job\': job, \'text\': chunk.decode(\'utf-8\', errors=\'replace\'),\n            \'offset\': offset, \'next_offset\': min(offset, size) + len(chunk),\n            \'total_bytes\': size, \'has_more\': min(offset, size) + len(chunk) < size}\n\n\n@app.post(\'/jobs/{job_id}/cancel\', dependencies=[Depends(secure)])\ndef cancel_job(job_id: str):\n    with LOCK:\n        process = PROCESSES.get(job_id)\n        if job_id not in JOBS:\n            raise HTTPException(status_code=404, detail=\'Unknown job ID\')\n        if not process or process.poll() is not None:\n            return {\'ok\': False, \'detail\': \'Job already finished\'}\n        JOBS[job_id][\'cancel_requested\'] = True\n    try:\n        os.killpg(process.pid, signal.SIGTERM)\n    except ProcessLookupError:\n        pass\n    return {\'ok\': True, \'job_id\': job_id}\n\n\n@app.get(\'/files/list\', dependencies=[Depends(secure)])\ndef files_list(path: str = \'/content\', limit: int = Query(default=100, ge=1, le=500)):\n    p = pathlib.Path(path).expanduser().resolve()\n    if not p.is_dir():\n        raise HTTPException(status_code=400, detail=\'Not a directory\')\n    try:\n        items = sorted(p.iterdir(), key=lambda f: (not f.is_dir(), f.name.lower()))[:limit]\n        return {\'path\': str(p), \'items\': [{\'name\': x.name, \'is_dir\': x.is_dir(), \'size_bytes\': x.stat().st_size if x.is_file() else None}\n                                           for x in items]}\n    except OSError as exc:\n        raise HTTPException(status_code=400, detail=str(exc))\n\n\n@app.get(\'/files/read\', dependencies=[Depends(secure)])\ndef files_read(path: str):\n    p = pathlib.Path(path).expanduser().resolve()\n    if not p.is_file():\n        raise HTTPException(status_code=404, detail=\'File not found\')\n    size = p.stat().st_size\n    if size > MAX_FILE:\n        raise HTTPException(status_code=413, detail=\'File too large for direct transfer; use shell commands or chunked transfer\')\n    return {\'path\': str(p), \'content_base64\': base64.b64encode(p.read_bytes()).decode(\'ascii\'), \'size_bytes\': size}\n\n\n@app.post(\'/files/write\', dependencies=[Depends(secure)])\ndef files_write(req: WriteRequest):\n    try:\n        data = base64.b64decode(req.content_base64, validate=True)\n    except Exception:\n        raise HTTPException(status_code=400, detail=\'Invalid base64 data\')\n    if len(data) > MAX_FILE:\n        raise HTTPException(status_code=413, detail=\'File too large for direct transfer\')\n    p = pathlib.Path(req.path).expanduser().resolve()\n    if req.create_parents:\n        p.parent.mkdir(parents=True, exist_ok=True)\n    p.write_bytes(data)\n    return {\'path\': str(p), \'size_bytes\': len(data)}\n\n\n@app.get(\'/recovery\', dependencies=[Depends(secure)])\ndef recovery():\n    # This endpoint is useful after transient tunnel failure while VM lives.\n    with LOCK:\n        recent = list(JOBS.values())[-20:]\n        events = list(EVENTS)\n    log_tails = {}\n    for j in recent[-8:]:\n        p = JOB_DIR / j[\'job_id\'] / \'output.log\'\n        if p.exists():\n            with p.open(\'rb\') as f:\n                f.seek(max(0, p.stat().st_size - 2000))\n                log_tails[j[\'job_id\']] = f.read().decode(\'utf-8\', errors=\'replace\')\n    return {\'status\': status(), \'recent_jobs\': recent, \'events\': events, \'recent_output_tails\': log_tails}\n\n\ndef restore_local_history():\n    for folder in JOB_DIR.glob(\'*/state.json\'):\n        try:\n            old = json.loads(folder.read_text(\'utf-8\'))\n            if old.get(\'status\') == \'running\':\n                old[\'status\'] = \'interrupted_or_unknown\'\n                old[\'finished_at\'] = now()\n            JOBS[old[\'job_id\']] = old\n        except Exception:\n            pass\n    event(\'bridge_started\', restored_jobs=len(JOBS))\n\n\nrestore_local_history()\nthreading.Thread(target=background_snapshots, daemon=True).start()\n'

data=base64.b64decode(PAYLOAD)
if hashlib.sha256(data).hexdigest()!=EXPECTED_SHA256:raise ValueError('Runtime tools package hash mismatch')
destination=pathlib.Path('/content').resolve()
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    if any(not (destination/name).resolve().is_relative_to(destination) for name in archive.namelist()):raise ValueError('Unsafe runtime archive')
    archive.extractall(destination)
del PAYLOAD,data
engine=destination/'video_engine';base=destination/'colab_codex_bridge';base.mkdir(exist_ok=True)
(base/'bridge_server.py').write_text(server_source,encoding='utf-8')
private=base/'bridge_private.json';saved=json.loads(private.read_text()) if private.is_file() else {}
token=saved.get('token');port=saved.get('port');server_pid=saved.get('server_pid')
reuse=False
if token and port and server_pid and psutil.pid_exists(server_pid):
    try:
        answer=requests.get(f'http://127.0.0.1:{port}/health',headers={'Authorization':'Bearer '+token},timeout=3)
        reuse=answer.status_code==200 and answer.json().get('api_version')==3
    except requests.RequestException:pass
if not reuse:
    if server_pid and psutil.pid_exists(server_pid):
        process=psutil.Process(server_pid)
        if 'bridge_server:app' in ' '.join(process.cmdline()):process.terminate();process.wait(timeout=10)
    token=secrets.token_urlsafe(48)
    with socket.socket() as sock:sock.bind(('127.0.0.1',0));port=sock.getsockname()[1]
    env=os.environ.copy();env.update(COLAB_BRIDGE_TOKEN=token,COLAB_BRIDGE_DIR=str(base),VIDEO_ENGINE_ROOT=str(engine))
    with (base/'server.log').open('ab',buffering=0) as output:
        server=subprocess.Popen([sys.executable,'-m','uvicorn','bridge_server:app','--host','127.0.0.1','--port',str(port),'--no-access-log'],cwd=base,env=env,stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
    server_pid=server.pid
    for attempt in range(30):
        if server.poll() is not None:raise RuntimeError('Runtime server exited. Inspect /content/colab_codex_bridge/server.log')
        try:
            if requests.get(f'http://127.0.0.1:{port}/health',headers={'Authorization':'Bearer '+token},timeout=2).status_code==200:break
        except requests.RequestException:pass
        time.sleep(1)
    else:raise RuntimeError('Runtime server did not become ready')
old_tunnel=saved.get('tunnel_pid')
if old_tunnel and psutil.pid_exists(old_tunnel):
    process=psutil.Process(old_tunnel)
    if 'cloudflared' in ' '.join(process.cmdline()):process.terminate()
cloudflared=base/'cloudflared'
if not cloudflared.is_file():
    if platform.machine().lower() not in ('x86_64','amd64'):raise RuntimeError('This launcher expects Linux x86_64')
    request=urllib.request.Request('https://api.github.com/repos/cloudflare/cloudflared/releases/latest',headers={'User-Agent':'VideoEngine/1.0'})
    metadata=json.loads(urllib.request.urlopen(request,timeout=30).read())
    asset=next(a for a in metadata['assets'] if a['name']=='cloudflared-linux-amd64')
    urllib.request.urlretrieve(asset['browser_download_url'],cloudflared)
    if asset.get('digest','').startswith('sha256:') and 'sha256:'+hashlib.sha256(cloudflared.read_bytes()).hexdigest()!=asset['digest']:raise ValueError('Cloudflared publisher digest mismatch')
    if cloudflared.read_bytes()[:4]!=b'\x7fELF':raise ValueError('Cloudflared is not a Linux executable')
    cloudflared.chmod(0o700)
tunnel=subprocess.Popen([str(cloudflared),'tunnel','--no-autoupdate','--protocol','http2','--url',f'http://127.0.0.1:{port}'],cwd=base,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
address={'url':None}
def consume():
    with (base/'tunnel.log').open('a') as output:
        for line in tunnel.stdout:
            output.write(line);output.flush()
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',line)
            if match:address['url']=match.group(0)
threading.Thread(target=consume,daemon=True).start()
for attempt in range(60):
    if address['url']:break
    if tunnel.poll() is not None:raise RuntimeError('Tunnel exited; inspect /content/colab_codex_bridge/tunnel.log')
    time.sleep(1)
else:raise RuntimeError('The tunnel did not return a URL')
private.write_text(json.dumps({'token':token,'port':port,'server_pid':server_pid,'tunnel_pid':tunnel.pid}));private.chmod(0o600)
print('\nGPU RUNTIME BRIDGE ONLINE',flush=True)
print('BRIDGE URL:',address['url'],flush=True)
print('PRIVATE TOKEN:',token,flush=True)
print('Keep the token private. This is a general runtime, not a prepared video.',flush=True)
print('Tools: Python, shell jobs, interactive terminal, large file upload/download, GPU project builds/probes/renders.',flush=True)
print('Preparing cached video tools; no scene or video is rendered during setup.',flush=True)
headers={'Authorization':'Bearer '+token}
setup_code='import subprocess,sys;subprocess.run([sys.executable,'+repr(str(engine/'runtime_setup.py'))+',"--root",'+repr(str(engine))+'],check=True)'
job=requests.post(f'http://127.0.0.1:{port}/jobs/python',headers=headers,json={'code':setup_code,'cwd':'/content','timeout_seconds':1800},timeout=10).json()
print('SETUP JOB:',job['job_id'],flush=True)
offset=0
while True:
    answer=requests.get(f'http://127.0.0.1:{port}/jobs/{job["job_id"]}/output',headers=headers,params={'offset':offset,'limit':160000},timeout=10).json()
    for line in answer['text'].splitlines():
        if line.startswith(('RUNTIME_','BUILD_STAGE','SNOW_','Traceback','RuntimeError')):print(line,flush=True)
    offset=answer['next_offset']
    if answer['job']['status']!='running':
        print('VIDEO TOOLS READY' if answer['job'].get('exit_code')==0 else 'Video setup failed; bridge remains available for diagnostics.',flush=True)
        break
    time.sleep(3)
print('Bridge is listening. Keep this cell running; stop it to close the endpoint.',flush=True)
sys.path.insert(0,str(engine))
from runtime_dashboard import render_dashboard
from IPython.display import HTML,display
dashboard=display(HTML(render_dashboard([])),display_id=True)
try:
    while psutil.pid_exists(server_pid) and tunnel.poll() is None:
        try:
            response=requests.get(f'http://127.0.0.1:{port}/jobs',headers=headers,timeout=10)
            response.raise_for_status()
            dashboard.update(HTML(render_dashboard(response.json())))
        except requests.RequestException:
            dashboard.update(HTML('<p>Progress connection interrupted; retrying. Existing job logs are retained.</p>'))
        time.sleep(3)
    print('Bridge server or tunnel stopped. Rerun this cell to reconnect.',flush=True)
except KeyboardInterrupt:
    tunnel.terminate()
    if psutil.pid_exists(server_pid):psutil.Process(server_pid).terminate()
    print('Runtime bridge stopped.',flush=True)
